##Data inspection
Can you predict if an individual will be able to make a payment in an emergency situation?

This notebook covers:
- Loading the data
- Simple EDA and an example of feature engineering
- Data preprocessing and data wrangling
- Creating a simple model
- Making a submission
- Some tips for improving your score

In [7]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

In [8]:
# Load files
train = pd.read_csv('data/Train.csv')
test = pd.read_csv('data/Test.csv')
samplesubmission = pd.read_csv('data/SampleSubmission.csv')
variable_definitions = pd.read_csv('data/VariableDefinitions.csv')
comments=pd.read_csv('data/Comments.csv')
competitionPartipation=pd.read_csv('data/CompetitionPartipation.csv')
competitions=pd.read_csv('data/Competitions.csv')
discussions=pd.read_csv('data/Discussions.csv')
submissions=pd.read_csv('data/Submissions.csv')
users=pd.read_csv('data/users.csv')
data_list = [train,test,samplesubmission,variable_definitions,comments,competitionPartipation,competitions,discussions,submissions,users]

In [9]:
def analyze(df):
    print('******************************************************************************')
    print(df.head())
    print(df.info())
    print(df.isna().sum())
#     for column in df.columns:
#         if(df[str(column)].dtype== 'int64'):
#             df[str(column)].plot()
#             plt.show()

    print('******************************************************************************')

In [10]:
for frame in data_list:
    analyze(frame)

******************************************************************************
       User_ID  month  year  CompPart  Comment  Sub  Disc  Target
0  ID_XI7BAR4Y      8     3         0        0    0     0       0
1  ID_XI7BAR4Y      8     2         0        0    0     0       0
2  ID_XI7BAR4Y      9     2         0        0    0     0       0
3  ID_XI7BAR4Y      9     3         0        0    0     0       0
4  ID_XI7BAR4Y     10     3         0        0    0     0       0
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 259832 entries, 0 to 259831
Data columns (total 8 columns):
 #   Column    Non-Null Count   Dtype 
---  ------    --------------   ----- 
 0   User_ID   259832 non-null  object
 1   month     259832 non-null  int64 
 2   year      259832 non-null  int64 
 3   CompPart  259832 non-null  int64 
 4   Comment   259832 non-null  int64 
 5   Sub       259832 non-null  int64 
 6   Disc      259832 non-null  int64 
 7   Target    259832 non-null  int64 
dtypes: int64(7), object(

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6211 entries, 0 to 6210
Data columns (total 6 columns):
 #   Column                Non-Null Count  Dtype 
---  ------                --------------  ----- 
 0   FeatureF              6211 non-null   int64 
 1   DiscDate Year         6211 non-null   int64 
 2   DiscDate Month        6211 non-null   int64 
 3   DiscDate Day_of_week  6211 non-null   int64 
 4   DiscID                6211 non-null   object
 5   UserID                6211 non-null   object
dtypes: int64(4), object(2)
memory usage: 291.3+ KB
None
FeatureF                0
DiscDate Year           0
DiscDate Month          0
DiscDate Day_of_week    0
DiscID                  0
UserID                  0
dtype: int64
******************************************************************************
******************************************************************************
        UserID  FeatureG   CompID  SubDate Year  SubDate Month  \
0  ID_8JP75F20         1  ID_GFDE           

## INSIGHTS

-Competition_Participation frame has two columns with missing data:

    1.Public Rank
    2.Successful Submission Count
    
-Competitions frame has five columns with missing data:

    1.Country
    2.Feature C
    3.Submission Limit Per Day
    4.CompEndTime Month
    5.CompEndTime Day of Week


In [11]:
def uniques(df):
    for column in df[df.columns]:
        print(column)
        print(df[column].unique())
        print(df[column].nunique())

In [12]:
for frame in data_list:
        print('******************************************************************************')
        uniques(frame)
        print('******************************************************************************')

******************************************************************************
User_ID
['ID_XI7BAR4Y' 'ID_TY9GL3X7' 'ID_ZOYZ41L5' ... 'ID_8IKU22O5' 'ID_NHWCR1IY'
 'ID_XMD7EIYV']
20218
month
[ 8  9 10 11 12  1  2  3  4  5  6  7]
12
year
[3 2 1]
3
CompPart
[0 1]
2
Comment
[0 1]
2
Sub
[0 1]
2
Disc
[0 1]
2
Target
[0 1]
2
******************************************************************************
******************************************************************************
User_ID
['ID_H1ELY25E' 'ID_463Q2BCO' 'ID_PSZU72Y3' ... 'ID_I3C1N5RO' 'ID_WWNR6I15'
 'ID_V69HAZHO']
22402
month
[1 2 3]
3
year
[4]
1
******************************************************************************
******************************************************************************
UserMonthYear
['ID_000VV0KM_1_4' 'ID_000VV0KM_2_4' 'ID_000VV0KM_3_4' ...
 'ID_ZZXDLYXB_1_4' 'ID_ZZXDLYXB_2_4' 'ID_ZZXDLYXB_3_4']
65223
Target
[nan]
0
******************************************************************************
****

[ 1.  2.  3.  4.  5.  6.  7.  8.  9. 10. nan 11. 12. 13. 14. 15. 16. 17.
 18. 19. 20. 21. 22. 23. 24. 25. 26. 27. 28. 29. 30. 31. 32. 33. 34. 35.
 36. 37.]
37
Kind
[1 0]
2
Points Reward
[ 27  52 126   2 102 151 500  77 599 550 301 251 350 211 226 114 400  99
 440 276]
20
FeatureD
[1 2 3]
3
SecretCode
[0 1]
2
SubmissionLimitPerDay
[100.  10. 200.  15.  30.  20.   3.  50.   5.   4.  nan  65.]
11
FeatureE
['[1]' '[2]' '[2, 3]' '[3, 2]' '[4]' '[1, 5]' '[6]' '[7]' '[3]' '[8]' '[]'
 '[9]' '[3, 5]' '[2, 1]' '[3, 1]' '[10]' '[1, 3]']
17
CompEndTime Year
['1' '4' '2' 'not mapped' '3']
5
CompEndTime Month
[ 8.  1. 11.  2. nan  6.  5.  9. 10. 12.  3.  4.  7.]
12
CompEndTime Day_of_week
[ 7.  6.  3. nan  1.  4.  5.  2.]
7
CompStartTime Year
[1 3 2 4]
4
CompStartTime Month
[ 6 12 10 11  2  1  3  9  4  7  5  8]
12
CompStartTime Day_of_week
[5 7 4 3 6 1 2]
7
******************************************************************************
*****************************************************************

In [15]:
train.Disc.sum()

3776